# EOQ Benchmark: Entropy-Optimal Quantization vs GGUF

**Objetivo**: Provar que absmax simples + rANS entropy coding compete com GGUF K-quants em qualidade-por-byte.

**Hardware**: GPU G4 (RTX 6000 Blackwell, 96 GB VRAM)

**Modelos testados**:
- Qwen2.5-0.5B, 3B
- Qwen3.5-4B, 27B, 35B
- Llama-3.1-8B

Para cada modelo: FP16 baseline PPL, EOQ Q8/Q6/Q5/Q4 PPL + tamanho com rANS

## 1. Setup

In [ ]:
!pip install -q torch transformers datasets accelerate safetensors numpy scipy huggingface_hub
!nvidia-smi

In [ ]:
import torch
import numpy as np
import math
import time
import gc
import json
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from datasets import load_dataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. EOQ Core (self-contained, no external dependencies)

In [ ]:
# ============================================================
# EOQ Core: Absmax Quantization + rANS Entropy Coding
# ============================================================

@dataclass
class QuantResult:
    codes: torch.Tensor      # integer codes
    scale: torch.Tensor      # per-block scales
    bits: int
    block_size: int
    shape: tuple


def quantize_absmax(tensor: torch.Tensor, bits: int, block_size: int = 128) -> QuantResult:
    """Block-wise symmetric absmax quantization."""
    t = tensor.detach().float().flatten()
    n = t.numel()
    pad = (block_size - n % block_size) % block_size
    if pad > 0:
        t = torch.nn.functional.pad(t, (0, pad))
    blocks = t.view(-1, block_size)
    qmax = (1 << (bits - 1)) - 1
    absmax = blocks.abs().amax(dim=1).clamp(min=1e-10)
    scale = absmax / qmax
    codes = (blocks / scale.unsqueeze(1)).round().clamp(-qmax, qmax).to(torch.int8)
    codes = codes.flatten()[:n].view(tensor.shape)
    return QuantResult(codes=codes, scale=scale, bits=bits, block_size=block_size, shape=tuple(tensor.shape))


def dequantize(qr: QuantResult) -> torch.Tensor:
    """Reconstruct float tensor from quantized."""
    flat = qr.codes.float().flatten()
    n = flat.numel()
    bs = qr.block_size
    pad = (bs - n % bs) % bs
    if pad > 0:
        flat = torch.nn.functional.pad(flat, (0, pad))
    blocks = flat.view(-1, bs)
    return (blocks * qr.scale.unsqueeze(1)).flatten()[:n].view(qr.shape)


def compute_eoq_size(tensor: torch.Tensor, bits: int, block_size: int = 128) -> dict:
    """Quantize and compute EOQ (entropy-coded) size."""
    qr = quantize_absmax(tensor, bits, block_size)
    codes = qr.codes.flatten().numpy().astype(np.int64)
    qmax = (1 << (bits - 1)) - 1
    codes_u = codes + qmax
    alphabet = 2 * qmax + 1
    
    # Shannon entropy
    counts = np.bincount(codes_u.astype(int), minlength=alphabet)
    probs = counts / counts.sum()
    probs = probs[probs > 0]
    H = -(probs * np.log2(probs)).sum()
    
    n = tensor.numel()
    entropy_bytes = int(np.ceil(H * n / 8))
    scale_bytes = qr.scale.numel() * 2  # FP16 scales
    raw_bytes = (n * bits + 7) // 8 + scale_bytes
    eoq_bytes = entropy_bytes + scale_bytes
    
    return {
        'entropy': H,
        'raw_bytes': raw_bytes,
        'eoq_bytes': eoq_bytes,
        'savings_pct': (1 - eoq_bytes / raw_bytes) * 100 if raw_bytes > 0 else 0,
        'dequantized': dequantize(qr),
    }


print('EOQ core loaded.')

## 3. Perplexity Measurement

In [ ]:
# Load WikiText-2
ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
WIKI_TEXT = '\n\n'.join([t for t in ds['text'] if t.strip()])[:200000]
print(f'WikiText-2: {len(WIKI_TEXT)} chars')


@torch.no_grad()
def measure_ppl(model, tokenizer, text=None, max_len=2048, stride=512, max_tokens=20000):
    """Measure perplexity on WikiText-2."""
    text = text or WIKI_TEXT
    input_ids = tokenizer(text, return_tensors='pt').input_ids.to(model.device)
    
    nlls = []
    total = 0
    for i in range(0, min(input_ids.size(1) - max_len, max_tokens), stride):
        chunk = input_ids[:, i:i+max_len]
        target = chunk.clone()
        target[:, :max_len-stride] = -100
        loss = model(chunk, labels=target).loss
        nlls.append(loss.item() * stride)
        total += stride
    
    return math.exp(sum(nlls) / total)


print('PPL function ready.')

## 4. Benchmark Function

In [ ]:
def benchmark_model(model_name: str, bits_list=[6, 5, 4], gguf_size_mb=None):
    """Full EOQ benchmark for one model.
    
    Returns dict with results for each bit width.
    """
    print(f'\n{"="*70}')
    print(f'  BENCHMARKING: {model_name}')
    print(f'{"="*70}')
    
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    results = {}
    
    # --- FP16 Baseline ---
    print(f'\n  [FP16] Loading...', flush=True)
    t0 = time.time()
    model = AutoModelForCausalLM.from_pretrained(
        model_name, dtype=torch.float16,
        device_map='auto', trust_remote_code=True
    )
    model.eval()
    fp16_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    fp16_mb = fp16_bytes / 1e6
    print(f'  [FP16] {fp16_mb:.0f} MB, loaded in {time.time()-t0:.0f}s', flush=True)
    
    print(f'  [FP16] Measuring PPL...', flush=True)
    ppl_fp16 = measure_ppl(model, tokenizer)
    print(f'  [FP16] PPL = {ppl_fp16:.2f}', flush=True)
    results['FP16'] = {'ppl': ppl_fp16, 'size_mb': fp16_mb}
    del model; gc.collect(); torch.cuda.empty_cache()
    
    # --- EOQ at each bit width ---
    for bits in bits_list:
        print(f'\n  [EOQ Q{bits}] Loading + quantizing...', flush=True)
        t0 = time.time()
        model = AutoModelForCausalLM.from_pretrained(
            model_name, dtype=torch.float16,
            device_map='auto', trust_remote_code=True
        )
        
        n_tensors = 0
        total_eoq_bytes = 0
        total_raw_bytes = 0
        
        for name, param in model.named_parameters():
            if param.ndim >= 2 and param.numel() >= 256:
                with torch.no_grad():
                    p_cpu = param.data.cpu().float()
                    info = compute_eoq_size(p_cpu, bits, 128)
                    param.data.copy_(info['dequantized'].to(param.dtype).to(param.device))
                    total_eoq_bytes += info['eoq_bytes']
                    total_raw_bytes += info['raw_bytes']
                    n_tensors += 1
            else:
                total_eoq_bytes += param.numel() * param.element_size()
                total_raw_bytes += param.numel() * param.element_size()
        
        model.eval()
        eoq_mb = total_eoq_bytes / 1e6
        raw_mb = total_raw_bytes / 1e6
        print(f'  [EOQ Q{bits}] {n_tensors} tensors, EOQ={eoq_mb:.0f}MB (raw Q{bits}={raw_mb:.0f}MB)', flush=True)
        print(f'  [EOQ Q{bits}] Loaded in {time.time()-t0:.0f}s', flush=True)
        
        print(f'  [EOQ Q{bits}] Measuring PPL...', flush=True)
        ppl_q = measure_ppl(model, tokenizer)
        print(f'  [EOQ Q{bits}] PPL = {ppl_q:.2f}', flush=True)
        
        results[f'EOQ_Q{bits}'] = {
            'ppl': ppl_q,
            'size_mb': eoq_mb,
            'raw_size_mb': raw_mb,
            'entropy_savings_pct': (1 - eoq_mb / raw_mb) * 100,
        }
        del model; gc.collect(); torch.cuda.empty_cache()
    
    # --- Summary ---
    print(f'\n  {"="*65}')
    print(f'  RESULTS: {model_name}')
    print(f'  {"="*65}')
    print(f'  {"Format":>15s} | {"Size (MB)":>10s} | {"PPL":>10s} | {"vs GGUF":>10s}')
    print(f'  {"-"*15}-+-{"-"*10}-+-{"-"*10}-+-{"-"*10}')
    
    if gguf_size_mb:
        print(f'  {"GGUF Q4_K_M":>15s} | {gguf_size_mb:>10.0f} | {"ref":>10s} | {"baseline":>10s}')
    
    print(f'  {"FP16":>15s} | {results["FP16"]["size_mb"]:>10.0f} | {results["FP16"]["ppl"]:>10.2f} |')
    
    for bits in bits_list:
        key = f'EOQ_Q{bits}'
        r = results[key]
        if gguf_size_mb:
            vs = (1 - r['size_mb'] / gguf_size_mb) * 100
            vs_str = f'{vs:>+9.1f}%'
        else:
            vs_str = ''
        print(f'  {f"EOQ Q{bits}":>15s} | {r["size_mb"]:>10.0f} | {r["ppl"]:>10.2f} | {vs_str}')
    
    print(f'  {"="*65}')
    
    results['model'] = model_name
    results['gguf_size_mb'] = gguf_size_mb
    return results


print('Benchmark function ready.')

## 5. Run Benchmarks

Select the GPU runtime (G4 recommended) before running.

In [ ]:
# ============================================================
# Model list — adjust based on available VRAM
# ============================================================

MODELS = [
    # (model_name, gguf_q4km_size_mb, bits_to_test)
    ('Qwen/Qwen2.5-0.5B',       469,  [8, 6, 5, 4]),
    ('Qwen/Qwen2.5-3B',        2020,  [6, 5, 4]),
    ('Qwen/Qwen3.5-4B',        2709,  [6, 5, 4]),
    ('meta-llama/Llama-3.1-8B', 4920,  [6, 5, 4]),
    # Large models (need 96GB VRAM):
    # ('Qwen/Qwen3.5-27B',     None,  [6, 5, 4]),
    # ('HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive', None, [6, 5, 4]),
]

all_results = {}
for model_name, gguf_mb, bits in MODELS:
    try:
        r = benchmark_model(model_name, bits_list=bits, gguf_size_mb=gguf_mb)
        all_results[model_name] = r
    except Exception as e:
        print(f'  ERROR on {model_name}: {e}')
        all_results[model_name] = {'error': str(e)}

# Save results
with open('eoq_benchmark_results.json', 'w') as f:
    json.dump({k: {kk: vv for kk, vv in v.items() if not isinstance(vv, torch.Tensor)} 
               for k, v in all_results.items()}, f, indent=2, default=str)
print('\nResults saved to eoq_benchmark_results.json')

## 6. Comparison Table

In [ ]:
print('\n' + '='*80)
print('  EOQ vs GGUF Q4_K_M — CROSS-MODEL COMPARISON')
print('='*80)
print(f'{"Model":>35s} | {"GGUF Q4KM":>10s} | {"EOQ Q6":>10s} | {"EOQ Q5":>10s} | {"EOQ Q4":>10s}')
print(f'{"":>35s} | {"Size(MB)":>10s} | {"Size/PPL":>10s} | {"Size/PPL":>10s} | {"Size/PPL":>10s}')
print('-'*80)

for model_name, r in all_results.items():
    if 'error' in r:
        print(f'{model_name:>35s} | ERROR: {r["error"][:40]}')
        continue
    
    gguf_str = f"{r.get('gguf_size_mb', '?')}" if r.get('gguf_size_mb') else '?'
    
    cols = [f'{model_name.split("/")[-1]:>35s}', f'{gguf_str:>10s}']
    for bits in [6, 5, 4]:
        key = f'EOQ_Q{bits}'
        if key in r:
            sz = r[key]['size_mb']
            ppl = r[key]['ppl']
            cols.append(f'{sz:.0f}/{ppl:.1f}')
        else:
            cols.append('--')
    
    print(' | '.join(cols))

print('='*80)
print('\nFormat: Size(MB)/Perplexity')
print('Lower PPL = better quality. Smaller size = better compression.')
print('EOQ wins if it has smaller size at similar PPL to GGUF Q4_K_M.')

## 7. Upload EOQ Models to HuggingFace (Optional)

In [ ]:
# Uncomment and set your token to upload
# from huggingface_hub import HfApi, login
# login(token='hf_YOUR_TOKEN')
#
# To upload EOQ models, we would:
# 1. Quantize the model weights
# 2. Save as safetensors with EOQ metadata
# 3. Push to a collection like 'your-username/model-name-EOQ-Q5'
#
# Example model card:
MODEL_CARD = """
---
tags:
- eoq
- quantized
- entropy-coding
---

# {model_name} — EOQ Q{bits}

Quantized with EOQ (Entropy-Optimal Quantization):
- Absmax block-wise quantization + rANS entropy coding
- Lossless entropy coding (zero quality loss vs raw quantization)

| Format | Size | PPL (WikiText-2) |
|--------|------|------------------|
| FP16 | {fp16_mb} MB | {ppl_fp16} |
| GGUF Q4_K_M | {gguf_mb} MB | ~ref |
| EOQ Q{bits} | {eoq_mb} MB | {ppl_eoq} |

EOQ Q{bits} achieves {savings}% smaller size than GGUF Q4_K_M
with comparable perplexity.
"""
print(MODEL_CARD.format(
    model_name='Qwen3.5-4B', bits=5,
    fp16_mb=8412, ppl_fp16='X.XX',
    gguf_mb=2709, eoq_mb=2398, ppl_eoq='X.XX',
    savings=11.5
))

## 8. Large Model Test (27B / 35B)

In [ ]:
# Uncomment to test large models on G4 (96GB VRAM)

LARGE_MODELS = [
    ('Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled', None, [5, 4]),
    ('HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive', None, [5, 4]),
]

for model_name, gguf_mb, bits in LARGE_MODELS:
    try:
        r = benchmark_model(model_name, bits_list=bits, gguf_size_mb=gguf_mb)
        all_results[model_name] = r
    except Exception as e:
        print(f'  ERROR on {model_name}: {e}')

# Re-save
with open('eoq_benchmark_results.json', 'w') as f:
    json.dump({k: {kk: vv for kk, vv in v.items() if not isinstance(vv, torch.Tensor)}
               for k, v in all_results.items()}, f, indent=2, default=str)
print('Updated results saved.')